# Is the RPI Fair? Code and live demo
**Every line of code we use is in this notebook.** Cell 1 copies the public repository (https://github.com/JiaheLi0604/apma4903-rpi-trueskill), which only supplies data: the 2022–2025 match table (from ncaavolleyballr) and the saved season predictions.

**Final model (Nov 20):** a team's expected lineup counts only players who have played this season (before the team's first match of a season: last season's players), and a new player starts at her team's current average rating instead of 25. The previous version is reproducible with `lineup='all', new_prior='global'`.

**Code style (Nov 21):** the model code (Algorithms 1–3, the season engine, 5c, 5d, 6a, 6b) is written with plain `for` loops; only the brute-force/Monte Carlo checks and the bootstrap use numpy arrays, for speed.

Run the cells in order. Sections 1–5 take a few seconds each. Section 6 (rebuilding the season predictions) takes several minutes and is optional.

In [ ]:
# Copy the public repository (data + saved predictions). Every line of CODE is in this notebook.
import os
if not os.path.exists('apma4903-rpi-trueskill'):
    !git clone -q https://github.com/JiaheLi0604/apma4903-rpi-trueskill.git
%cd apma4903-rpi-trueskill

## 1. The algorithm: Algorithm 1 (MessagePassing, one match)
Gaussians are stored in precision form (pi = 1/var, tau = mu/var), so multiplying adds and dividing subtracts. All the code is written with plain `for` loops and lists (Intro-to-Python style). Each `# line k` comment matches line k of the Algorithm 1 slide.

In [ ]:
import math
import numpy as np
from scipy.special import log_ndtr, ndtr

LOG_SQRT_2PI = 0.9189385332046727          # log of sqrt(2*pi)


def VW(t):
    """Nugget 2's functions.  V(t) = phi(t) / Phi(t),  W(t) = V(t) * (V(t) + t).
    phi(t) / Phi(t) is computed as exp(log phi(t) - log Phi(t)), so it does not break for very negative t."""
    log_phi = -0.5 * t * t - LOG_SQRT_2PI        # log of the standard normal PDF at t
    log_Phi = log_ndtr(t)                         # log of the standard normal CDF at t
    V = math.exp(log_phi - log_Phi)
    W = V * (V + t)
    return V, W


def message_passing(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True, log=None, empty_team=None):
    """Algorithm 1 (MessagePassing): one match.
    Input : for each player of team A and team B: mean mu, variance var, weight w;
            beta (performance noise); h (home term for team A: +home advantage if A is at home,
            -home advantage if B is at home, 0 at a neutral site); A_won (True or False).
    Output: (new means of A, new variances of A), (new means of B, new variances of B).
    If `log` is a list, every message is added to it as (name, mean, variance), only for printing.
    empty_team = (mu_T, v_T): the team-level Gaussian used for a team with NO recorded players
    (its first match ever, Notation slide). That team enters the prediction (lines 1-2) but has no players to update."""

    beta2 = beta * beta

    # ---- winner W and loser L, and the home term eta from the winner's side
    if A_won:
        names_W, names_L = "A", "B"
        mu_W, var_W, w_W = list(muA), list(varA), list(wA)
        mu_L, var_L, w_L = list(muB), list(varB), list(wB)
        eta = h
    else:
        names_W, names_L = "B", "A"
        mu_W, var_W, w_W = list(muB), list(varB), list(wB)
        mu_L, var_L, w_L = list(muA), list(varA), list(wA)
        eta = -h

    # ================= DOWN: predict the gap =================
    # line 1: a_i = w_i * mu_i,  b_i = w_i^2 * (sigma_i^2 + beta^2);  mu_T = sum of a_i,  v_T = sum of b_i
    a_W = []
    b_W = []
    for i in range(len(mu_W)):
        a_W.append(w_W[i] * mu_W[i])
        b_W.append(w_W[i] ** 2 * (var_W[i] + beta2))
    a_L = []
    b_L = []
    for i in range(len(mu_L)):
        a_L.append(w_L[i] * mu_L[i])
        b_L.append(w_L[i] ** 2 * (var_L[i] + beta2))

    mu_TW = 0.0
    v_TW = 0.0
    for i in range(len(a_W)):
        mu_TW = mu_TW + a_W[i]
        v_TW = v_TW + b_W[i]
    mu_TL = 0.0
    v_TL = 0.0
    for i in range(len(a_L)):
        mu_TL = mu_TL + a_L[i]
        v_TL = v_TL + b_L[i]

    # a team with no recorded players: use its team-level Gaussian (mu_T, v_T); the loop below then has nobody to update
    if len(mu_W) == 0:
        mu_TW = empty_team[0]
        v_TW = empty_team[1]
    if len(mu_L) == 0:
        mu_TL = empty_team[0]
        v_TL = empty_team[1]

    # line 2: the gap d = winner - loser:  m = mu_W - mu_L + eta,  c^2 = v_W + v_L,  t = m / c
    m = mu_TW - mu_TL + eta
    c2 = v_TW + v_TL
    c = math.sqrt(c2)
    t = m / c

    # ================= CUT: the winner won, so d > 0 (Nugget 2) =================
    # line 3: m_hat = m + c * V(t),  v_hat = c^2 * (1 - W(t))
    V, W = VW(t)
    m_hat = m + c * V
    v_hat = c2 * (1 - W)

    # ================= UP: send only the evidence back =================
    # line 4: divide out the prediction:  pi_e = 1/v_hat - 1/c^2,  m_e = (m_hat/v_hat - m/c^2) / pi_e
    pi_e = 1.0 / v_hat - 1.0 / c2
    m_e = (m_hat / v_hat - m / c2) / pi_e

    # line 5: message to the winning team:  team W = d + team L
    mu_to_TW = m_e + mu_TL - eta
    v_to_TW = 1.0 / pi_e + v_TL
    # line 6: message to the losing team:   team L = team W - d
    mu_to_TL = mu_TW + eta - m_e
    v_to_TL = 1.0 / pi_e + v_TW

    # lines 7-9: for every player of both teams
    new = {}
    teams = [(names_W, mu_W, var_W, w_W, a_W, b_W, mu_TW, v_TW, mu_to_TW, v_to_TW),
             (names_L, mu_L, var_L, w_L, a_L, b_L, mu_TL, v_TL, mu_to_TL, v_to_TL)]
    for team in teams:
        name, mu, var, w, a, b, mu_T, v_T, mu_to_T, v_to_T = team
        new_mu = []
        new_var = []
        for i in range(len(mu)):
            # line 8: take the teammates out and divide by her weight, then add her own beta^2
            #         (mu_T - a_i is the teammates' part of the team mean)
            mu_to_i = (mu_to_T - mu_T + a[i]) / w[i]
            v_to_i = (v_to_T + v_T - b[i]) / w[i] ** 2 + beta2
            # line 9: multiply into her old rating: precisions add
            pi_i = 1.0 / var[i] + 1.0 / v_to_i
            mu_i = (mu[i] / var[i] + mu_to_i / v_to_i) / pi_i
            var_i = 1.0 / pi_i
            new_mu.append(mu_i)
            new_var.append(var_i)
            if log is not None:
                log.append((f"what the match says about {name}{i+1}", mu_to_i, v_to_i))
                log.append((f"new rating {name}{i+1}", mu_i, var_i))
        new[name] = (np.array(new_mu), np.array(new_var))

    if log is not None:
        log.insert(0, ("team " + names_W + " (winner), predicted", mu_TW, v_TW))
        log.insert(1, ("team " + names_L + " (loser), predicted", mu_TL, v_TL))
        log.insert(2, ("gap d, predicted", m, c2))
        log.insert(3, ("gap d after the cut (Nugget 2)", m_hat, v_hat))
        log.insert(4, ("news about d (prediction divided out)", m_e, 1.0 / pi_e))
        log.insert(5, ("message to team " + names_W, mu_to_TW, v_to_TW))
        log.insert(6, ("message to team " + names_L, mu_to_TL, v_to_TL))

    return new["A"], new["B"]

## 2. The two independent checks
`closed_form`: Nugget 2's update formula, with player weights.
`brute_force_grid`: Bayes' rule computed directly on a grid (prior × P(A wins | skills)); no Gaussian approximation.
`monte_carlo`: the same Bayes' rule with random samples, for 6 vs. 6 where a grid is too large.

In [ ]:
from scipy.special import ndtr

def closed_form(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True):
    """Nugget 2's formulas, written for weighted teams (road 2 of the verification).
    Each player moves by  +/- w_i * sigma_i^2 / c * V(t)  and her variance shrinks by the factor
    1 - w_i^2 * sigma_i^2 / c^2 * W(t)."""
    beta2 = beta * beta
    mean_A = 0.0
    var_A = 0.0
    for i in range(len(muA)):
        mean_A = mean_A + wA[i] * muA[i]
        var_A = var_A + wA[i] ** 2 * (varA[i] + beta2)
    mean_B = 0.0
    var_B = 0.0
    for i in range(len(muB)):
        mean_B = mean_B + wB[i] * muB[i]
        var_B = var_B + wB[i] ** 2 * (varB[i] + beta2)
    mu_d = mean_A - mean_B + h
    c2 = var_A + var_B
    c = math.sqrt(c2)
    if A_won:
        sign = 1.0
    else:
        sign = -1.0
    V, W = VW(sign * mu_d / c)
    newA_mu = []
    newA_var = []
    for i in range(len(muA)):
        newA_mu.append(muA[i] + sign * wA[i] * varA[i] / c * V)
        newA_var.append(varA[i] * (1 - wA[i] ** 2 * varA[i] / c2 * W))
    newB_mu = []
    newB_var = []
    for i in range(len(muB)):
        newB_mu.append(muB[i] - sign * wB[i] * varB[i] / c * V)
        newB_var.append(varB[i] * (1 - wB[i] ** 2 * varB[i] / c2 * W))
    return (np.array(newA_mu), np.array(newA_var)), (np.array(newB_mu), np.array(newB_var))


# ---- The two checks below are written with numpy arrays on purpose: they add up millions of grid
# ---- points or random samples, which plain Python loops would take minutes to do.
# ---- They are checks only; the model itself (Algorithms 1 and 2) never uses them.

def brute_force_grid(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True, n=None, half_width=7.0):
    """Exact posterior mean/var of every skill by direct numerical integration on a grid (Bayes' rule).
    The performance noise is integrated analytically: P(A wins | skills) = Phi((sum wA sA - sum wB sB + h)/sqrt(beta^2 sum w^2)).
    n grid values per skill over mean +/- 7 sd: 801 for 1 vs 1, 61 for 2 vs 2."""
    mus = np.r_[muA, muB]; sds = np.sqrt(np.r_[varA, varB]); ws = np.r_[wA, -np.asarray(wB, float)]
    k = len(mus)
    if n is None:
        if k == 2:
            n = 801
        elif k == 4:
            n = 61
        else:
            n = 41
    axes = []
    for j in range(k):
        axes.append(np.linspace(mus[j] - half_width * sds[j], mus[j] + half_width * sds[j], n))
    grids = np.meshgrid(*axes, indexing="ij", sparse=True)      # every combination of grid values
    logprior = 0.0
    diff = h
    for j in range(k):
        logprior = logprior - 0.5 * ((grids[j] - mus[j]) / sds[j]) ** 2      # log of the Gaussian prior
        diff = diff + ws[j] * grids[j]                                          # sum wA sA - sum wB sB + h
    if A_won:
        sign = 1.0
    else:
        sign = -1.0
    lik = ndtr(sign * diff / np.sqrt(beta ** 2 * np.sum(ws ** 2)))             # P(result | skills)
    wgt = np.exp(logprior - logprior.max()) * lik                               # prior x likelihood
    Z = wgt.sum()
    means = []
    variances = []
    for j in range(k):
        m = (wgt * grids[j]).sum() / Z
        means.append(m)
        variances.append((wgt * (grids[j] - m) ** 2).sum() / Z)
    k1 = len(muA)
    return (np.array(means[:k1]), np.array(variances[:k1])), (np.array(means[k1:]), np.array(variances[k1:]))


def monte_carlo(muA, varA, wA, muB, varB, wB, beta, h=0.0, A_won=True, N=4_000_000, seed=0):
    """Same exact posterior, by sampling skills from the prior and weighting each sample by P(result | skills)."""
    rng = np.random.default_rng(seed)
    mus = np.r_[muA, muB]; sds = np.sqrt(np.r_[varA, varB]); ws = np.r_[wA, -np.asarray(wB, float)]
    S = mus + sds * rng.standard_normal((N, len(mus)))          # N random draws of every skill
    if A_won:
        sign = 1.0
    else:
        sign = -1.0
    lik = ndtr(sign * (S @ ws + h) / np.sqrt(beta ** 2 * np.sum(ws ** 2)))
    Z = lik.sum()
    m = (lik @ S) / Z
    v = (lik @ (S - m) ** 2) / Z
    k1 = len(muA)
    return (m[:k1], v[:k1]), (m[k1:], v[k1:])

## 3. The 2 vs. 2 toy match
Change any number and rerun this cell.

In [ ]:
beta = 25/6        # match-day noise
h = 0.0            # home advantage for team A (0 = neutral site)
A_won = True

#         means      variances (sigma^2)   weights
muA, varA, wA = [28, 22], [3**2, 6**2], [1, 1]
muB, varB, wB = [26, 24], [4**2, 2**2], [1, 1]

log = []
(mA, vA), (mB, vB) = message_passing(muA, varA, wA, muB, varB, wB, beta, h, A_won, log)
print("Every message (mean, sd):")
for entry in log:
    name = entry[0]
    m = entry[1]
    v = entry[2]
    print(f"  {name:40s} {m:8.2f} {math.sqrt(v):6.2f}")

print("\nPlayer   before          after")
for i in range(len(muA)):
    print(f"  A{i+1}   {muA[i]:5.2f} ± {math.sqrt(varA[i]):4.2f}   {mA[i]:5.2f} ± {math.sqrt(vA[i]):4.2f}")
for i in range(len(muB)):
    print(f"  B{i+1}   {muB[i]:5.2f} ± {math.sqrt(varB[i]):4.2f}   {mB[i]:5.2f} ± {math.sqrt(vB[i]):4.2f}")


def max_difference(x, y):
    """Largest gap between two results, over every player's mean and standard deviation."""
    (xA_mu, xA_var), (xB_mu, xB_var) = x
    (yA_mu, yA_var), (yB_mu, yB_var) = y
    biggest = 0.0
    for i in range(len(xA_mu)):
        biggest = max(biggest, abs(xA_mu[i] - yA_mu[i]), abs(math.sqrt(xA_var[i]) - math.sqrt(yA_var[i])))
    for i in range(len(xB_mu)):
        biggest = max(biggest, abs(xB_mu[i] - yB_mu[i]), abs(math.sqrt(xB_var[i]) - math.sqrt(yB_var[i])))
    return biggest


mp_out = ((mA, vA), (mB, vB))
cf = closed_form(muA, varA, wA, muB, varB, wB, beta, h, A_won)
bf = brute_force_grid(muA, varA, wA, muB, varB, wB, beta, h, A_won)
print(f"\nmax difference vs. Nugget 2 formula: {max_difference(mp_out, cf):.1e}")
print(f"max difference vs. grid integration: {max_difference(mp_out, bf):.1e}")

## 4. Verification table: three methods on four test matches

In [ ]:
import numpy as np
beta = 25/6
#        name                      muA      varA          wA         muB      varB          wB        h    A_won
cases = [("1v1 two newcomers",       [25],    [8.333**2],   [1],       [25],    [8.333**2],   [1],      0,   True),
         ("1v1 veteran vs newcomer", [25],    [2.0**2],     [1],       [25],    [8.333**2],   [1],      0,   True),
         ("1v1 upset (weak wins)",   [20],    [3.0**2],     [1],       [30],    [3.0**2],     [1],      0,   True),
         ("2v2 weighted, home",      [26,22], [3**2,6**2],  [1.2,0.8], [27,24], [4**2,2**2],  [1.0,1.0], 2.0, False)]

summary = []
for case in cases:
    name, mA, vA, wA, mB, vB, wB, h, won = case
    mp_ = message_passing(mA, vA, wA, mB, vB, wB, beta, h, won)
    cf = closed_form(mA, vA, wA, mB, vB, wB, beta, h, won)
    bf = brute_force_grid(mA, vA, wA, mB, vB, wB, beta, h, won)
    print(f"\n== {name}")
    for side, k in (("A", 0), ("B", 1)):
        for i in range(len(mp_[k][0])):
            print(f"  {side}{i+1}: MP mu={mp_[k][0][i]:.4f} sd={math.sqrt(mp_[k][1][i]):.4f}"
                  f" | CF mu={cf[k][0][i]:.4f} sd={math.sqrt(cf[k][1][i]):.4f}"
                  f" | GRID mu={bf[k][0][i]:.4f} sd={math.sqrt(bf[k][1][i]):.4f}")
    print(f"  max |MP-CF| = {max_difference(mp_, cf):.2e}")
    summary.append((name, max_difference(mp_, bf)))
print()
for name, e in summary:
    print(f"{name:26s} max |MP - grid| = {e:.1e}")

## 5. Live run: UTEP vs. North Carolina, Dec 4, 2025
First the season engine (5a) and the lineup recorder (5b) are defined: they produce every player's rating and weight before the match. Then 5c runs the match (about 15 seconds).

### 5a. Season engine: TrueSkill walked forward through every match (Algorithm 2)
Line 10 calls `message_passing` from Section 1 for every match. A team with no recorded players (its first match ever) enters the prediction with its team-level Gaussian and has no players to update.

In [ ]:
"""Season engine (Algorithm 2, SeasonScore): player-level TrueSkill with playing-time weights, walked forward day by day.
Written with plain for loops so every step can be read line by line.  Lines marked "Algorithm 2, line k" match the slide."""
import math
import numpy as np
import pandas as pd
from scipy.special import log_ndtr, ndtr
# message_passing (Algorithm 1) is defined in Section 1 above; line 10 of Algorithm 2 calls it.
LOG_SQRT_2PI = 0.9189385332046727          # log of sqrt(2*pi)
W_BAR = 6.0                                 # a volleyball team always has six players on the court

DEFAULT_PARAMS = {"mu0": 25.0, "sigma0": 25.0 / 3, "beta": 25.0 / 6, "tau_season": 25.0 / 300,
                  "home": 0.0, "decay": 0.6, "rho": 1.0}


def v_w(t):
    """Nugget 2's V(t) = phi(t) / Phi(t) and W(t) = V(t) * (V(t) + t), computed in log space."""
    log_phi = -0.5 * t * t - LOG_SQRT_2PI
    v = math.exp(log_phi - log_ndtr(t))
    w = v * (v + t)
    if w < 0.0:                      # rounding guard: W is always between 0 and 1
        w = 0.0
    if w > 1.0 - 1e-12:
        w = 1.0 - 1e-12
    return v, w


def prepare_data(pm, matches):
    """Turn the cleaned box scores (pm) and the match table (matches) into integer-indexed arrays."""
    agg = pm.groupby(["Team", "Date", "Player"], as_index=False, dropna=False)["S"].sum()   # sets per player per day
    team_names = sorted(set(pm["Team"]) | set(matches["A"]) | set(matches["B"]))
    team_id = {t: i for i, t in enumerate(team_names)}
    agg["tid"] = agg["Team"].map(team_id).astype(int)
    agg["pid"], player_keys = pd.factorize(agg["tid"].astype(str) + "||" + agg["Player"].astype(str))
    date_list = sorted(set(agg["Date"]) | set(matches["Date"]))
    date_idx = {d: i for i, d in enumerate(date_list)}
    agg["di"] = agg["Date"].map(date_idx).astype(int)

    box = {}                                       # box[day][team] = (player ids, sets played)
    for (di, tid), g in agg.groupby(["di", "tid"]):
        box.setdefault(di, {})[tid] = (g["pid"].values.astype(int), g["S"].values.astype(float))

    season_of = pm.groupby("Date")["SeasonYear"].first()
    season_of_day = {date_idx[d]: int(s) for d, s in season_of.items() if d in date_idx}

    m_di = matches["Date"].map(date_idx).astype(int).values
    m_by_day = {}
    for i, di in enumerate(m_di):
        m_by_day.setdefault(di, []).append(i)

    return dict(
        n_players=len(player_keys), n_teams=len(team_names), team_names=team_names, team_id=team_id,
        player_keys=player_keys, date_list=date_list, box=box, season_of_day=season_of_day,
        days=sorted(set(box) | set(m_by_day)), m_by_day=m_by_day, n_matches=len(matches),
        A=matches["A"].map(team_id).astype(int).values, B=matches["B"].map(team_id).astype(int).values,
        A_won=matches["A_won"].values.astype(int), Home=matches["Home"].values.astype(float),
        m_season=matches["SeasonYear"].values.astype(int), m_di=m_di)



def on_court_weights(share):
    """Rescale playing-time shares so that the team's weights add up to 6 (Algorithm 2, line 7)."""
    total = 0.0
    for x in share:
        total = total + x
    weights = []
    for x in share:
        weights.append(W_BAR * x / total)
    return np.array(weights)


def run_walk_forward(D, P, level="player", new_prior="team", lineup="active", end_season=None,
                     fallback=(6.0, 1.9), return_lineups=False):
    """Algorithm 2 (SeasonScore).  Walk through every day in order.  For each day:
         (0) register players seen for the first time (Algorithm 2, line 5),
         (1) predict that day's matches from beliefs built ONLY from earlier days (lines 7-9),
         (2) update beliefs with that day's results (line 10, Algorithm 1: message passing),
         (3) add that day's sets played to the playing-time memory (lines 11-12).
       level      : 'player' (team = weighted sum of its players)  or  'team' (each team is one 'player')
       new_prior  : 'team'   a new player starts at her team's current weighted mean rating (final model)
                    'global' a new player starts at 25, as in TrueSkill
       lineup     : 'active' only players who have played this season; before the team's first box score of a
                             season, the players who played last season (final model)
                    'all'    everyone who has ever played for the team
       Returns a dictionary; out['p'][m] = predicted probability that team A wins match m."""
    mu0 = P["mu0"]
    sig0 = P["sigma0"]
    beta = P["beta"]
    tau = P["tau_season"]
    h = P["home"]
    decay = P["decay"]
    rho = P["rho"]
    tau_m2 = P.get("tau_match", 0.0) ** 2           # optional drift before every match (only used by one baseline)
    beta2 = beta * beta

    if level == "player":
        n_units = D["n_players"]
    else:
        n_units = D["n_teams"]

    # ---- the state the engine keeps (Algorithm 2, line 1)
    mu = np.full(n_units, float(mu0))               # rating mean of every player
    var = np.full(n_units, float(sig0) ** 2)        # rating variance of every player
    seen = np.zeros(n_units, bool)                  # has this player appeared yet?
    sets = np.zeros(n_units)                        # n_i: sets she has played (older seasons discounted)
    team_sets = np.zeros(D["n_teams"])              # N_T: sets her team has played (older seasons discounted)
    last_season = np.full(n_units, -1)              # last season in which she played (before today)
    team_played = np.zeros(D["n_teams"], bool)      # has the team played this season (before today)?
    roster = []                                     # roster[team] = every player who has ever played for the team
    for team in range(D["n_teams"]):
        roster.append([])

    n = D["n_matches"]
    p_out = np.full(n, np.nan)
    muA_out = np.full(n, np.nan)
    muB_out = np.full(n, np.nan)
    c_out = np.full(n, np.nan)
    lineups = None
    if return_lineups:
        lineups = [None] * n
    fb_mu = fallback[0] * mu0                       # a team with no history: six average players
    fb_V = fallback[1] * (sig0 ** 2 + beta2)

    cur_season = None

    def expected_lineup(team):
        """Algorithm 2, line 7: the players of the team's lineup and their weights, from earlier days only."""
        if level == "team":
            return [team], [1.0]
        if team_sets[team] <= 0:
            return [], []
        ids = []
        shares = []
        for i in roster[team]:
            share = sets[i] / team_sets[team]               # r_i = n_i / N_T
            if share > 1.0:
                share = 1.0                                 # r_i = min(n_i / N_T, 1)
            if share > 0:
                ids.append(i)
                shares.append(share)
        if lineup == "active":                              # keep only this season's players
            if team_played[team]:
                oldest_ok = cur_season                      # the team has played this season
            else:
                oldest_ok = cur_season - 1                  # before its first match: last season's players
            ids_active = []
            shares_active = []
            for k in range(len(ids)):
                if last_season[ids[k]] >= oldest_ok:
                    ids_active.append(ids[k])
                    shares_active.append(shares[k])
            if len(ids_active) > 0:                         # (if nobody qualifies, keep everyone)
                ids = ids_active
                shares = shares_active
        if len(ids) == 0:
            return [], []
        return ids, list(on_court_weights(shares))         # w_i = 6 r_i / sum of r_j

    def team_mean_var(ids, weights):
        """Algorithm 1, line 1: team mean = sum w_i mu_i,  team variance = sum w_i^2 (sigma_i^2 + beta^2)."""
        if len(ids) == 0:
            return fb_mu, fb_V
        m = 0.0
        v = 0.0
        for k in range(len(ids)):
            m = m + weights[k] * mu[ids[k]]
            v = v + weights[k] ** 2 * (var[ids[k]] + beta2)
        return m, v

    for di in D["days"]:                                    # Algorithm 2, line 2: for every day d
        season = D["season_of_day"].get(di, cur_season)
        if end_season is not None and season is not None and season > end_season:
            break

        # ---- Algorithm 2, lines 3-4: a new season starts
        if cur_season is not None and season != cur_season:
            for i in range(n_units):
                sets[i] = sets[i] * decay                   # n_i <- decay * n_i
                if seen[i]:
                    mu[i] = mu0 + rho * (mu[i] - mu0)       # pull toward 25 by rho
                    var[i] = var[i] + tau ** 2              # add tau^2
            for team in range(D["n_teams"]):
                team_sets[team] = team_sets[team] * decay   # N_T <- decay * N_T
                team_played[team] = False
        cur_season = season
        todays_box = D["box"].get(di, {})                   # box score: who played how many sets today
        todays_matches = D["m_by_day"].get(di, [])

        # ---- Algorithm 2, line 5: first appearances
        if level == "player":
            for team in todays_box:
                ids_today = todays_box[team][0]
                new_players = []
                for i in ids_today:
                    if not seen[i]:
                        new_players.append(i)
                if len(new_players) > 0:
                    start = mu0                             # 'global': start at 25
                    if new_prior == "team":                 # 'team': start at the team's current weighted mean
                        ids, weights = expected_lineup(team)
                        if len(ids) > 0:
                            total_w = 0.0
                            total_wmu = 0.0
                            for k in range(len(ids)):
                                total_w = total_w + weights[k]
                                total_wmu = total_wmu + weights[k] * mu[ids[k]]
                            start = total_wmu / total_w
                    for i in new_players:
                        seen[i] = True
                        mu[i] = start
                        var[i] = sig0 ** 2
                        roster[team].append(i)
        else:
            for m in todays_matches:
                for team in (D["A"][m], D["B"][m]):
                    if not seen[team]:
                        seen[team] = True
                        mu[team] = mu0
                        var[team] = sig0 ** 2

        # ---- Algorithm 2, lines 6-10: predict, then update, every match of the day
        for m in todays_matches:
            a = D["A"][m]
            b = D["B"][m]
            ids_a, w_a = expected_lineup(a)                 # line 7: weights from earlier days only
            ids_b, w_b = expected_lineup(b)
            if return_lineups:
                lineups[m] = ((np.array(ids_a, dtype=int), np.array(w_a)), (np.array(ids_b, dtype=int), np.array(w_b)))
            if tau_m2 > 0:
                for i in ids_a:
                    var[i] = var[i] + tau_m2
                for i in ids_b:
                    var[i] = var[i] + tau_m2
            mean_a, v_a = team_mean_var(ids_a, w_a)
            mean_b, v_b = team_mean_var(ids_b, w_b)
            c2 = v_a + v_b
            mu_d = mean_a - mean_b + h * D["Home"][m]       # line 8: predicted gap (A - B), with home advantage
            p_out[m] = ndtr(mu_d / math.sqrt(c2))           # line 9: P(A wins), scored BEFORE the update
            muA_out[m] = mean_a
            muB_out[m] = mean_b
            c_out[m] = math.sqrt(c2)
            # line 10: Algorithm 1 (message passing) on this match.  A team with no recorded players
            # enters with its team-level Gaussian (fb_mu, fb_V) and has nobody to update.
            mu_a = []
            var_a = []
            for i in ids_a:
                mu_a.append(mu[i])
                var_a.append(var[i])
            mu_b = []
            var_b = []
            for i in ids_b:
                mu_b.append(mu[i])
                var_b.append(var[i])
            (new_mu_a, new_var_a), (new_mu_b, new_var_b) = message_passing(
                mu_a, var_a, w_a, mu_b, var_b, w_b, beta, h * D["Home"][m], D["A_won"][m] == 1,
                empty_team=(fb_mu, fb_V))
            for k in range(len(ids_a)):
                mu[ids_a[k]] = new_mu_a[k]
                var[ids_a[k]] = new_var_a[k]
            for k in range(len(ids_b)):
                mu[ids_b[k]] = new_mu_b[k]
                var[ids_b[k]] = new_var_b[k]

        # ---- Algorithm 2, lines 11-12: add today's sets played, once per day
        if level == "player":
            for team in todays_box:
                ids_today, sets_today = todays_box[team]
                most = 0.0
                for k in range(len(ids_today)):
                    sets[ids_today[k]] = sets[ids_today[k]] + sets_today[k]
                    last_season[ids_today[k]] = cur_season
                    if sets_today[k] > most:
                        most = sets_today[k]
                team_sets[team] = team_sets[team] + most        # sets_T(d) = max over players of sets_i(d)
                team_played[team] = True

    return {"p": p_out, "muA": muA_out, "muB": muB_out, "c": c_out, "mu": mu, "var": var, "sets": sets,
            "team_sets": team_sets, "roster": roster, "seen": seen, "last_season": last_season,
            "team_played": team_played, "lineups": lineups}


# ------------------------------------------------------------------ scores
def log_loss(p, y):
    """Mean of -log(probability given to what happened).  Lower is better; a coin flip scores 0.693."""
    total = 0.0
    for k in range(len(p)):
        q = min(max(p[k], 1e-6), 1 - 1e-6)
        if y[k] == 1:
            total = total - math.log(q)
        else:
            total = total - math.log(1 - q)
    return total / len(p)


def accuracy(p, y):
    """Share of matches where the favourite (probability > 0.5) won."""
    right = 0
    for k in range(len(p)):
        if (p[k] > 0.5) == (y[k] == 1):
            right = right + 1
    return right / len(p)


def brier(p, y):
    total = 0.0
    for k in range(len(p)):
        total = total + (p[k] - y[k]) ** 2
    return total / len(p)

### 5b. Lineup recorder: each player's playing-time weight before each match (the engine's own lineups)


In [ ]:
from math import sqrt

def record_lineups(D, P):
    """For every match, the players of each side and their weights, exactly as the season engine (5a) used them
    for its prediction: past sets only, active lineup (players who have played this season)."""
    return run_walk_forward(D, P, return_lineups=True)["lineups"]


# ------------------------------------------------------------------ 2. build the graph of player-time nodes

### 5c. The match

In [ ]:
# 5c. The match: UTEP vs. North Carolina, Dec 4, 2025
import pickle
import pandas as pd
D = pickle.load(open('work/D.pkl', 'rb'))
P = pickle.load(open('work/P_tuned.pkl', 'rb'))
names = D['team_names']
day = pd.Timestamp('2025-12-04')

# find the match
i = None
for j in range(D['n_matches']):
    team_a = names[D['A'][j]]
    team_b = names[D['B'][j]]
    if D['date_list'][D['m_di'][j]] == day and {team_a, team_b} == {'UTEP', 'North Carolina'}:
        i = j
        break
di = D['m_di'][i]
print('match', i, 'A=', names[D['A'][i]], 'B=', names[D['B'][i]], 'A_won', D['A_won'][i], 'Home', D['Home'][i])

# every rating the night before: run the season engine (5a) on the days before the match only
D2 = dict(D)
D2['days'] = []
for d in D['days']:
    if d < di:
        D2['days'].append(d)
o = run_walk_forward(D2, P)
mu = o['mu']
var = o['var']
(ia, wa), (ib, wb) = record_lineups(D, P)[i]          # the lineups and weights the engine used for this match
keys = D['player_keys']


def show(ids, w, label):
    print(label)
    order = sorted(range(len(ids)), key=lambda k: -w[k])        # largest weight first
    for k in order:
        p = ids[k]
        print(f"   {keys[p].split('||')[1]:22s} w={w[k]:.2f} mu={mu[p]:6.2f} sd={math.sqrt(var[p]):.2f}")


show(ia, wa, names[D['A'][i]])
show(ib, wb, names[D['B'][i]])

beta = P["beta"]
h = 0.0          # neutral site (played in Madison); the data wrongly marks it as a UTEP home game
won = bool(D['A_won'][i])

# Algorithm 1, line 1 by hand: team means and the spread c of the gap
mA = 0.0
vA = 0.0
for k in range(len(ia)):
    mA = mA + wa[k] * mu[ia[k]]
    vA = vA + wa[k] ** 2 * (var[ia[k]] + beta ** 2)
mB = 0.0
vB = 0.0
for k in range(len(ib)):
    mB = mB + wb[k] * mu[ib[k]]
    vB = vB + wb[k] ** 2 * (var[ib[k]] + beta ** 2)
c = math.sqrt(vA + vB)
print(f"team means {mA:.1f} vs {mB:.1f}, c={c:.2f}, P(A wins)={ndtr((mA - mB + h) / c):.3f}, h={h}")

# the update three ways: message passing, Nugget 2's formula, Monte Carlo (3 million samples)
mu_a, var_a, mu_b, var_b = mu[ia], var[ia], mu[ib], var[ib]
log = []
mp_out = message_passing(mu_a, var_a, wa, mu_b, var_b, wb, beta, h, won, log)
cf_out = closed_form(mu_a, var_a, wa, mu_b, var_b, wb, beta, h, won)
mc_out = monte_carlo(mu_a, var_a, wa, mu_b, var_b, wb, beta, h, won, N=3_000_000)
print('max|MP-CF|', max_difference(mp_out, cf_out))
print('max|MP-MC| (mu or sd)', max_difference(mp_out, mc_out))
for entry in log:
    if entry[0].startswith(('team', 'gap', 'news')):
        print(f"  {entry[0]:40s} mean={entry[1]:8.3f} sd={math.sqrt(entry[2]):.3f}")

### 5d. Show how the weights are calculated (North Carolina, before Dec 4)
Step 1: each player's share of the team's sets in earlier matches (old seasons multiplied by decay). Step 2: keep only players who have played this season. Step 3: rescale so the team's weights sum to 6. This is exactly the lineup the engine (5a) uses for the prediction.


In [ ]:
# 5d. How the weights are calculated: North Carolina, before the Dec 4, 2025 match
import pickle
import pandas as pd
D = pickle.load(open('work/D.pkl', 'rb'))
P = pickle.load(open('work/P_tuned.pkl', 'rb'))
team = D['team_id']['North Carolina']
cut = pd.Timestamp('2025-12-04')                      # only matches BEFORE this day are used
D2 = dict(D)
D2['days'] = []
for di in D['days']:
    if D['date_list'][di] < cut:
        D2['days'].append(di)
o = run_walk_forward(D2, P)                           # the engine's playing-time memory the night before

rows = []
for p in o['roster'][team]:                           # everyone who has ever played for the team
    share = o['sets'][p] / o['team_sets'][team]       # step 1: share of the team's sets, r = n_i / N_T
    if share > 1.0:
        share = 1.0                                   #         r = min(n_i / N_T, 1)
    if share > 0 and o['last_season'][p] >= 2025:     # step 2: has played this season
        rows.append([D['player_keys'][p].split('||')[1], o['sets'][p], share])

total = 0.0
for row in rows:
    total = total + row[2]
for row in rows:
    row.append(6 * row[2] / total)                    # step 3: rescale so the team sums to 6

rows.sort(key=lambda row: -row[3])                    # largest weight first
print(f"{'player':22s} {'sets (decayed)':>15s} {'raw share':>10s} {'weight':>8s}")
for row in rows[:12]:
    print(f"{row[0]:22s} {row[1]:15.3f} {row[2]:10.3f} {row[3]:8.3f}")
weight_sum = 0.0
for row in rows:
    weight_sum = weight_sum + row[3]
print(f"\nplayers ever on the roster: {len(o['roster'][team])}   played this season: {len(rows)}   team sets (decayed): {o['team_sets'][team]:.1f}")
print(f"sum of raw shares: {total:.2f}   ->   rescaled by 6 / {total:.2f}   ->   sum of weights: {weight_sum:.2f}")

## 6. Results: TrueSkill vs. the RPI on the 2025 season
Three columns on the results slides:
- **RPI**: recomputed every day from this season's results (6c).
- **TrueSkill, same information**: the same algorithm with one rating per team (the 1 vs. 1 case), this season's wins and losses only, no home court (6c, 6d). It sees exactly what the RPI sees.
- **TrueSkill, full**: the 6 vs. 6 player-level model from Nugget 3 (final model: active lineup, new players start at their team's average; `work/P_tuned.pkl`, tuned on 2024 in section 7). It is recomputed below in a few seconds.

6a and 6b take seconds. 6c and 6d rebuild the RPI and same-information predictions from scratch (several minutes each); you do not need to run them. Needs 5a to have been run.

### 6a. Accuracy, log-loss, bootstrap intervals

In [ ]:
# 6a. Accuracy, log-loss, bootstrap intervals
import pickle
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.optimize import minimize_scalar
D = pickle.load(open('work/D.pkl', 'rb'))
R = pickle.load(open('cmp/rpi_wf.pkl', 'rb'))
T = pickle.load(open('cmp/ts_wf.pkl', 'rb'))
n = D['n_matches']
y = D['A_won']
S = D['m_season']
dates = []
for m in range(n):
    dates.append(D['date_list'][D['m_di'][m]])
dR = R['rpi_A'] - R['rpi_B']                          # RPI gap (A minus B) before each match
k = np.minimum(R['nA'], R['nB'])                      # matches played so far by the team with fewer


def loss(p, won):
    """-log(probability given to what happened) for one match."""
    p = min(max(p, 1e-12), 1 - 1e-12)
    if won == 1:
        return -math.log(p)
    return -math.log(1 - p)


def right(p, won):
    """1 if the favourite won, else 0."""
    if (p > 0.5) == (won == 1):
        return 1.0
    return 0.0


# turn the RPI gap into a probability, P(A wins) = expit(slope * gap), slope fitted on 2024
fit_ids = []
for m in range(n):
    if S[m] == 2024 and k[m] >= 5:
        fit_ids.append(m)


def rpi_loss(slope):
    total = 0.0
    for m in fit_ids:
        total = total + loss(expit(slope * dR[m]), y[m])
    return total / len(fit_ids)


slope = minimize_scalar(rpi_loss, bounds=(0, 200), method='bounded').x

# TrueSkill full = the player-level model from Nugget 3, walked forward through every match (a few seconds)
P_player = pickle.load(open('work/P_tuned.pkl', 'rb'))
p_player = run_walk_forward(D, P_player)['p']
M = {'RPI': expit(slope * dR), 'RPI+home': T['rpi_home'], 'TS same info': T['fair_tuned'], 'TS full': p_player}

rng = np.random.default_rng(1)


def ci(d):
    """95% bootstrap interval of the mean of d: resample the matches 2000 times.
    (numpy here: 2000 resamples of ~3,500 matches is too slow with plain loops.)"""
    d = np.array(d)
    means = []
    for b in range(2000):
        means.append(d[rng.integers(0, len(d), len(d))].mean())
    return np.round(np.percentile(means, [2.5, 97.5]), 4)


subsets = [('2025 both>=5', None, None),
           ('2025 before Oct15', None, pd.Timestamp('2025-10-15')),
           ('2025 Oct15+', pd.Timestamp('2025-10-15'), None),
           ('Tourney', pd.Timestamp('2025-12-04'), None)]
for label, start, end in subsets:
    ids = []
    for m in range(n):
        if S[m] != 2025 or np.isnan(dR[m]):
            continue
        if label != 'Tourney' and k[m] < 5:
            continue
        if start is not None and dates[m] < start:
            continue
        if end is not None and dates[m] >= end:
            continue
        ids.append(m)
    print('==', label, 'n=', len(ids))
    for name in M:
        p = M[name]
        total_loss = 0.0
        total_right = 0.0
        for m in ids:
            total_loss = total_loss + loss(p[m], y[m])
            total_right = total_right + right(p[m], y[m])
        print(f'  {name:14s} LL={total_loss / len(ids):.4f} acc={total_right / len(ids):.4f}')
    for a, c in [('TS same info', 'RPI'), ('TS full', 'RPI'), ('TS full', 'RPI+home')]:
        d_loss = []
        d_acc = []
        disagree = 0
        a_right = 0
        for m in ids:
            d_loss.append(loss(M[a][m], y[m]) - loss(M[c][m], y[m]))
            d_acc.append(right(M[a][m], y[m]) - right(M[c][m], y[m]))
            if (M[a][m] > 0.5) != (M[c][m] > 0.5):           # the two models pick different winners
                disagree = disagree + 1
                a_right = a_right + right(M[a][m], y[m])
        print(f'  {a} - {c}: dLL={np.mean(d_loss):+.4f} CI={ci(d_loss)} dAcc={np.mean(d_acc):+.4f} CI={ci(d_acc)};'
              f' disagree n={disagree} {a} right {a_right / disagree:.3f}')

### 6b. Rankings at the end of the regular season, and UTEP vs. North Carolina (a few seconds)

In [ ]:
# Rankings at the end of the regular season, and the UTEP vs. North Carolina predictions (TrueSkill full = player-level)
import pickle, numpy as np, pandas as pd
from scipy.special import expit
from scipy.stats import spearmanr, norm
from scipy.optimize import minimize_scalar
D = pickle.load(open('work/D.pkl', 'rb'))
T = pickle.load(open('cmp/ts_wf.pkl', 'rb'))
R = pickle.load(open('cmp/rpi_wf.pkl', 'rb'))
A = D['A']
B = D['B']
y = D['A_won']
S = D['m_season']
DI = D['m_di']
dates = D['date_list']
names = D['team_names']
nT = D['n_teams']

def rpi_from(games):
    """RPI = 0.25 WP + 0.50 OWP + 0.25 OOWP from a list of (team a, team b, a won?) games."""
    W = np.zeros(nT)
    N = np.zeros(nT)
    pw = {}
    pn = {}
    for a, b, aw in games:                                  # WP: wins and games for every team
        W[a] = W[a] + aw
        W[b] = W[b] + 1 - aw
        N[a] = N[a] + 1
        N[b] = N[b] + 1
        pw[(a, b)] = pw.get((a, b), 0) + aw          # a's wins against b
        pw[(b, a)] = pw.get((b, a), 0) + 1 - aw
        pn[(a, b)] = pn.get((a, b), 0) + 1           # games between a and b
        pn[(b, a)] = pn.get((b, a), 0) + 1
    with np.errstate(invalid='ignore', divide='ignore'):
        wp = W / N
    os_ = np.zeros(nT)
    oc = np.zeros(nT)
    # OWP: opponents' win %, leaving out games vs. this team
    for a, b, aw in games:
        for t, o in ((a, b), (b, a)):
            n = N[o] - pn[(o, t)]
            if n > 0:
                os_[t] = os_[t] + (W[o] - pw[(o, t)]) / n
                oc[t] = oc[t] + 1
    with np.errstate(invalid='ignore', divide='ignore'):
        owp = os_ / oc
    s2 = np.zeros(nT)
    c2 = np.zeros(nT)
    # OOWP: opponents' OWP
    for a, b, aw in games:
        for t, o in ((a, b), (b, a)):
            if not np.isnan(owp[o]):
                s2[t] = s2[t] + owp[o]
                c2[t] = c2[t] + 1
    with np.errstate(invalid='ignore', divide='ignore'):
        oowp = s2 / c2
    return 0.25*wp + 0.5*owp + 0.25*oowp, N, (wp, owp, oowp)

cut = pd.Timestamp('2025-11-30')                              # end of the regular season
games = []
for i in range(len(A)):
    if S[i] == 2025 and dates[DI[i]] < cut:
        games.append((A[i], B[i], int(y[i])))
rpi, N, (wp, owp, oowp) = rpi_from(games)


def days_before(day, season=None):
    """The list of day indices before `day` (and, if given, only in `season`)."""
    out = []
    for di in D['days']:
        if dates[di] < day and (season is None or D['season_of_day'][di] == season):
            out.append(di)
    return out


def lineup(o, t, season=2025):
    """The team's expected lineup, exactly as the engine (5a) builds it: r = min(n_i / N_T, 1),
    only players who have played this season (before the team's first match: last season's), weights sum to 6."""
    if o['team_sets'][t] <= 0:
        return [], []
    if o['team_played'][t]:
        oldest_ok = season
    else:
        oldest_ok = season - 1
    ids_all, r_all, ids_act, r_act = [], [], [], []
    for p in o['roster'][t]:
        r = min(o['sets'][p] / o['team_sets'][t], 1.0)
        if r > 0:
            ids_all.append(p)
            r_all.append(r)
            if o['last_season'][p] >= oldest_ok:
                ids_act.append(p)
                r_act.append(r)
    if len(ids_act) > 0:
        return ids_act, list(on_court_weights(r_act))
    if len(ids_all) > 0:
        return ids_all, list(on_court_weights(r_all))
    return [], []


def team_mean_var(o, t, beta2):
    """Algorithm 1, line 1: sum of w_i mu_i and sum of w_i^2 (sigma_i^2 + beta^2)."""
    ids, w = lineup(o, t)
    if len(ids) == 0:
        return float('nan'), float('nan')
    m = 0.0
    v = 0.0
    for k in range(len(ids)):
        m = m + w[k] * o['mu'][ids[k]]
        v = v + w[k] ** 2 * (o['var'][ids[k]] + beta2)
    return m, v


# TrueSkill, full = the player-level model. A team's strength is its expected lineup, sum of w_i * mu_i
P_player = pickle.load(open('work/P_tuned.pkl', 'rb'))
b2 = P_player['beta'] ** 2
D2 = dict(D)
D2['days'] = days_before(cut)
oF = run_walk_forward(D2, P_player)
muF = []
for t in range(nT):
    muF.append(team_mean_var(oF, t, b2)[0])
D3 = dict(D)
D3['days'] = days_before(cut, season=2025)
muS = run_walk_forward(D3, T['P_fair'], level='team')['mu']      # TrueSkill same info: this season only, one rating per team

# the table (pandas only to sort and rank)
rows = []
for t in range(nT):
    if N[t] > 0:
        wins = int(round(wp[t] * N[t]))
        rows.append({'team': names[t], 'W-L': f"{wins}-{int(N[t] - wins)}", 'n': N[t],
                     'WP': wp[t], 'OWP': owp[t], 'OOWP': oowp[t], 'RPI': rpi[t], 'same': muS[t], 'full': muF[t]})
df = pd.DataFrame(rows)
df['RPI rank'] = df['RPI'].rank(ascending=False).astype(int)
df['TS same info'] = df['same'].rank(ascending=False).astype(int)
df['TS full'] = df['full'].rank(ascending=False).astype(int)
show = df[df.team.isin(['UTEP', 'North Carolina'])].set_index('team')
print(show[['W-L', 'WP', 'OWP', 'OOWP', 'RPI', 'RPI rank', 'TS same info', 'TS full']].round(3).to_string())

# Largest disagreements: among teams with at least 20 D1 matches and in either top 64, three largest gaps each way
cand = df[(df['n'] >= 20) & ((df['RPI rank'] <= 64) | (df['TS full'] <= 64))].copy()
cand['gap'] = cand['TS full'] - cand['RPI rank']
cols = ['team', 'W-L', 'RPI rank', 'TS same info', 'TS full']
print('\nRPI ranks higher than TrueSkill full:')
print(cand.nlargest(3, 'gap')[cols].to_string(index=False))
print('TrueSkill full ranks higher than the RPI:')
print(cand.nsmallest(3, 'gap')[cols].to_string(index=False))
print(f"\nRank correlation with the RPI: TrueSkill full {spearmanr(df.RPI, df.full, nan_policy='omit').correlation:.2f},"
      f" same info {spearmanr(df.RPI, df.same).correlation:.2f}")

# Pre-match P(North Carolina wins), Dec 4, 2025, neutral site
for j in range(len(A)):
    if S[j] == 2025 and {names[A[j]], names[B[j]]} == {'UTEP', 'North Carolina'}:
        i = j
        break
unc_is_A = names[A[i]] == 'North Carolina'

# RPI: P(A wins) = expit(slope * RPI gap), slope fitted on 2024 (same as 6a)
dR = R['rpi_A'] - R['rpi_B']
k = np.minimum(R['nA'], R['nB'])
fit_ids = []
for m in range(len(A)):
    if S[m] == 2024 and k[m] >= 5 and not np.isnan(dR[m]):
        fit_ids.append(m)


def rpi_loss(slope):
    total = 0.0
    for m in fit_ids:
        p = min(max(expit(slope * dR[m]), 1e-12), 1 - 1e-12)
        if y[m] == 1:
            total = total - math.log(p)
        else:
            total = total - math.log(1 - p)
    return total / len(fit_ids)


slope = minimize_scalar(rpi_loss, bounds=(0, 200), method='bounded').x
p_rpi = expit(slope * dR[i])
p_same = T['fair_tuned'][i]

D4 = dict(D)
D4['days'] = days_before(pd.Timestamp('2025-12-04'))
o = run_walk_forward(D4, P_player)
mA, vA = team_mean_var(o, D['team_id']['North Carolina'], b2)
mB, vB = team_mean_var(o, D['team_id']['UTEP'], b2)
p_full_unc = norm.cdf((mA - mB) / math.sqrt(vA + vB))            # h = 0, neutral site (same as 5c)
if unc_is_A:
    p_rpi_unc, p_same_unc = p_rpi, p_same
else:
    p_rpi_unc, p_same_unc = 1 - p_rpi, 1 - p_same
print(f"\nP(North Carolina wins):  RPI {p_rpi_unc:.0%}   TrueSkill same info {p_same_unc:.0%}   TrueSkill full {p_full_unc:.0%}")

### 6c. (optional, slow) Rebuild: walk-forward RPI and TrueSkill predictions

In [ ]:
import sys,pickle,numpy as np,pandas as pd
D=pickle.load(open('work/D.pkl','rb')); P=pickle.load(open('work/P_team.pkl','rb'))
A,B,y,S,DI,H=D['A'],D['B'],D['A_won'].astype(bool),D['m_season'],D['m_di'],D['Home']
dates=D['date_list']; nT=D['n_teams']
print('Home values',np.unique(H,return_counts=True))
# ---------- walk-forward RPI (season to date, D1 vs D1, before the match day) ----------
rpi_A=np.full(len(A),np.nan); rpi_B=np.full(len(A),np.nan); nA=np.zeros(len(A)); nB=np.zeros(len(A))
def rpi_from(games):
    W=np.zeros(nT); N=np.zeros(nT); pw={}; pn={}
    for a,b,aw in games:
        W[a]+=aw; W[b]+=1-aw; N[a]+=1; N[b]+=1
        pw[(a,b)]=pw.get((a,b),0)+aw; pw[(b,a)]=pw.get((b,a),0)+1-aw
        pn[(a,b)]=pn.get((a,b),0)+1; pn[(b,a)]=pn.get((b,a),0)+1
    with np.errstate(invalid='ignore',divide='ignore'): wp=W/N
    os=np.zeros(nT); oc=np.zeros(nT)
    for a,b,aw in games:
        for t,o in ((a,b),(b,a)):
            n=N[o]-pn[(o,t)]
            if n>0: os[t]+=(W[o]-pw[(o,t)])/n; oc[t]+=1
    with np.errstate(invalid='ignore',divide='ignore'): owp=os/oc
    s2=np.zeros(nT); c2=np.zeros(nT)
    for a,b,aw in games:
        for t,o in ((a,b),(b,a)):
            if not np.isnan(owp[o]): s2[t]+=owp[o]; c2[t]+=1
    with np.errstate(invalid='ignore',divide='ignore'): oowp=s2/c2
    return 0.25*wp+0.5*owp+0.25*oowp, N, (wp,owp,oowp)
for season in (2024,2025):
    days=sorted(set(DI[S==season]))
    games=[]
    for di in days:
        idx=np.where(DI==di)[0]
        if games:
            r,N,_=rpi_from(games)
            rpi_A[idx]=r[A[idx]]; rpi_B[idx]=r[B[idx]]; nA[idx]=N[A[idx]]; nB[idx]=N[B[idx]]
        games+= [(A[i],B[i],int(y[i])) for i in idx]
    if season==2025:
        final_rpi,_,comps=rpi_from([g for g in games if dates[DI[0]]<=dates[DI[0]]])
pickle.dump(dict(rpi_A=rpi_A,rpi_B=rpi_B,nA=nA,nB=nB),open('cmp/rpi_wf.pkl','wb'))
# ---------- TrueSkill team-level: full and "fair" (current season only, no home) ----------
full=run_walk_forward(D,P,level='team')['p']
Pf=dict(P,home=0.0)
fair=np.full(len(A),np.nan)
for season in (2024,2025):
    D2=dict(D); D2['days']=[di for di in D['days'] if D['season_of_day'][di]==season]
    fair_s=run_walk_forward(D2,Pf,level='team')['p']; m=S==season; fair[m]=fair_s[m]
pickle.dump(dict(full=full,fair=fair),open('cmp/ts_wf.pkl','wb'))
print('done')

### 6d. (optional, slow) Tune same-information TrueSkill; add a home-court term to the RPI

In [ ]:
import sys,pickle,numpy as np
from scipy.special import expit
from scipy.optimize import minimize
D=pickle.load(open('work/D.pkl','rb')); P=pickle.load(open('work/P_team.pkl','rb')); R=pickle.load(open('cmp/rpi_wf.pkl','rb'))
y=D['A_won'].astype(float); S=D['m_season']; k=np.minimum(R['nA'],R['nB']); H=D['Home']
def ll(p,yy): p=np.clip(p,1e-12,1-1e-12); return -np.mean(yy*np.log(p)+(1-yy)*np.log(1-p))
def fair(Pf,season):
    D2=dict(D); D2['days']=[di for di in D['days'] if D['season_of_day'][di]==season]
    return run_walk_forward(D2,Pf,level='team')['p']
m24=(S==2024)&(k>=5)
res=[]
for s0 in [2,3,4,5,6,8.33,12]:
  for tm in [0,0.5,1]:
    Pf=dict(P,home=0.0,sigma0=s0,tau_match=tm); p=fair(Pf,2024); res.append((ll(p[m24],y[m24]),s0,tm))
res.sort(); print(res[:5])
best=res[0]; Pf=dict(P,home=0.0,sigma0=best[1],tau_match=best[2])
p25=fair(Pf,2025); p24=fair(Pf,2024)
# RPI + home logistic fitted on 2024
dR=R['rpi_A']-R['rpi_B']
f=lambda w: ll(expit(w[0]*dR[m24]+w[1]*H[m24]),y[m24])
w=minimize(f,[16,0.2]).x; print('RPI+home weights',w)
pRH=expit(w[0]*dR+w[1]*H)
T=pickle.load(open('cmp/ts_wf.pkl','rb')); T['fair_tuned']=np.where(S==2025,p25,p24); T['rpi_home']=pRH; T['P_fair']=Pf
pickle.dump(T,open('cmp/ts_wf.pkl','wb'))

## 7. Learning the settings by maximum likelihood: Algorithm 3 (CoordinateSearch) — about 3 minutes
Objective (Nugget 1): maximize $L(\theta)=\sum_{\text{2024 matches}}\log P(\text{result}\mid\text{earlier results},\theta)$; mean log loss $=-L/n$, so the lowest log loss is the maximum likelihood.
Method (Algorithm 3): coordinate search, one setting at a time over its fixed grid, moving only on a strict improvement, repeated until a full round changes nothing. Each evaluation of $L$ is one SeasonScore run (Algorithm 2 = `run_walk_forward` from 5a): ratings run 2022–2024, only 2024 matches are scored, each predicted before it is used; 2025 is never touched. $\beta$ is fixed because it only sets the scale.
This is an approximate E-step plus coordinate search, not EM, so there is no EM ascent guarantee; the search itself never lowers the $L$ it computes. The likelihood is fairly flat near the optimum: a finer grid and a gradient method (7b) find slightly higher validation likelihoods with different settings. Needs 5a to have been run.

In [ ]:
# 7. Algorithm 3: learn the settings by maximum likelihood (loop to convergence), about 3 minutes
import pickle, sys, time
import numpy as np
D = pickle.load(open("work/D.pkl", "rb"))
y = D["A_won"]
val = D["m_season"] == 2024          # True for every 2024 match (validation season)
tst = D["m_season"] == 2025          # True for every 2025 match (test season)

grids = {"sigma0": [2, 3, 4, 5, 6, 8.33, 10, 12],
         "tau_season": [0, 0.25, 0.5, 1, 1.5, 2, 3],
         "home": [0, 0.5, 1, 1.5, 2, 3, 4],
         "decay": [0.2, 0.4, 0.6, 0.8, 1.0],
         "rho": [0.6, 0.8, 0.9, 1.0]}

runs = []                            # every SeasonScore execution (one full walk through 2022-2024)


def neg_ll(P):
    """SeasonScore: mean negative log-likelihood (log-loss) of the 2024 predictions under settings P."""
    setting = []
    for name in grids:
        setting.append(P[name])
    runs.append(tuple(setting))
    out = run_walk_forward(D, P, end_season=2024)
    return log_loss(out["p"][val], y[val])


def describe(P):
    text = ""
    for name in grids:
        text = text + f"{name}={P[name]:g}  "
    return text


P = dict(DEFAULT_PARAMS)
t0 = time.time()
cur = neg_ll(P)
print(f"start (TrueSkill defaults): log-loss {cur:.5f}")
it = 0
while True:                          # Algorithm 3: repeat full rounds until nothing changes
    it = it + 1
    changed = []
    for name in grids:               # one parameter at a time
        best_ll = None
        best_v = None
        for v in grids[name]:        # try every grid value
            Q = dict(P)
            Q[name] = v
            ll = neg_ll(Q)
            if best_ll is None or ll < best_ll:
                best_ll = ll
                best_v = v
        if best_ll < cur:            # strict improvement only (Algorithm 3, line 6)
            if best_v != P[name]:
                changed.append(f"{name} {P[name]:g}->{best_v:g}")
            P[name] = best_v
            cur = best_ll
    if len(changed) > 0:
        print(f"round {it}: log-loss {cur:.5f}  {describe(P)}  changed: {changed}")
    else:
        print(f"round {it}: log-loss {cur:.5f}  {describe(P)}  (no change: converged)")
        break

distinct = []                        # count the distinct settings tried
for r in runs:
    if r not in distinct:
        distinct.append(r)
print(f"converged after {it} rounds: {len(runs)} SeasonScore runs ({len(distinct)} distinct settings), {time.time()-t0:.0f} s")

out = run_walk_forward(D, P)
for label, mask in (("validation 2024", val), ("test 2025", tst)):
    p = out["p"][mask]
    print(f"{label}: log-loss {log_loss(p, y[mask]):.4f}  accuracy {accuracy(p, y[mask]):.1%}  (all {mask.sum()} matches)")
try:
    old = pickle.load(open("work/P_tuned.pkl", "rb"))
    same = True
    for name in grids:
        if abs(old[name] - P[name]) > 1e-12:
            same = False
    print("same as P_tuned.pkl:", same)
except FileNotFoundError:
    pass

### 7b. (optional) How flat is the likelihood near the choice? A gradient method and a finer grid
L-BFGS-B (finite-difference gradients) on the same 2024 log-loss, from the TrueSkill defaults (about 1–2 minutes), then a finer grid of 540 combinations around the choice (several minutes). Both find slightly better 2024 scores (by 0.0002–0.0004) with different settings: the predictive likelihood is fairly flat around several good solutions, so we do not claim the methods recover the same θ.


In [ ]:
# 7b-1. gradient method (L-BFGS-B), about 1-2 minutes
import pickle, time, numpy as np
from scipy.optimize import minimize
D = pickle.load(open('work/D.pkl', 'rb')); y = D['A_won']; val = D['m_season'] == 2024; tst = D['m_season'] == 2025
P_grid = pickle.load(open('work/P_tuned.pkl', 'rb'))
names = ['sigma0', 'tau_season', 'home', 'decay', 'rho']; lo = [1, 0, 0, 0.05, 0.3]; hi = [15, 3, 5, 1, 1]
def P_of(x): P = dict(DEFAULT_PARAMS); P.update(dict(zip(names, x))); return P
n = [0]
def f(x):
    n[0] += 1; return log_loss(run_walk_forward(D, P_of(x), end_season=2024)['p'][val], y[val])
def test(x): o = run_walk_forward(D, P_of(x)); return log_loss(o['p'][tst], y[tst])
t0 = time.time()
x0 = np.array([DEFAULT_PARAMS[k] for k in names])
r = minimize(f, x0, method='L-BFGS-B', bounds=list(zip(lo, hi)), options=dict(eps=1e-3, maxiter=40))
print('L-BFGS-B from defaults:', dict(zip(names, np.round(r.x, 3))), ' 2024 log-loss %.5f  2025 log-loss %.5f' % (r.fun, test(r.x)),
      ' iterations', r.nit, 'evaluations', n[0], round(time.time() - t0), 's')
xg = np.array([P_grid[k] for k in names])
fg = f(xg); print('coordinate-search choice:', dict(zip(names, xg)), ' 2024 log-loss %.5f  2025 log-loss %.5f' % (fg, test(xg)))
g = []
for i in range(5):
    e = np.zeros(5); e[i] = 1e-2
    g.append((fg - f(xg - e)) / 1e-2 if xg[i] + 1e-2 > hi[i] else (f(xg + e) - f(xg - e)) / 2e-2)
print('finite-difference gradient of the 2024 log-loss at the choice:', dict(zip(names, np.round(g, 5))))


In [ ]:
# 7b-2. finer grid of 540 combinations, several minutes
import sys, pickle, itertools, time, numpy as np
from multiprocessing import Pool
D = pickle.load(open('work/D.pkl', 'rb')); P0 = pickle.load(open('work/P_tuned.pkl', 'rb'))
mv = D['m_season'] == 2024; mt = D['m_season'] == 2025; y = D['A_won']
S = [3, 3.5, 4, 4.5, 5, 6]; H = [2, 2.5, 3, 3.5, 4]; DE = [0.6, 0.8, 1.0]; T = [0.5, 1, 1.5]; R = [0.9, 1.0]
combos = list(itertools.product(S, H, DE, T, R))           # 6 x 5 x 3 x 3 x 2 = 540
def f(c):
    s, h, de, t, r = c; P = dict(P0, sigma0=s, home=h, decay=de, tau_season=t, rho=r)
    p = run_walk_forward(D, P)["p"]
    return c, log_loss(p[mv], y[mv]), log_loss(p[mt], y[mt])
if __name__ == "__main__":
    t0 = time.time()
    with Pool() as pool: res = pool.map(f, combos)
    pickle.dump(res, open('lec/grid_res.pkl', 'wb'))
    res.sort(key=lambda r: r[1])
    print(len(res), "combinations (sigma0, home, decay, tau, rho)", round(time.time() - t0), "s")
    for r in res[:5]: print("  ", r[0], "2024 %.5f  2025 %.5f" % (r[1], r[2]))
    pick = (P0['sigma0'], P0['home'], P0['decay'], P0['tau_season'], P0['rho'])
    cs = [r for r in res if r[0] == pick][0]
    print("coordinate-search pick", pick, "2024 %.5f  2025 %.5f" % (cs[1], cs[2]), "rank", res.index(cs) + 1, "of", len(res),
          " 2024 gap to #1: %.5f" % (cs[1] - res[0][1]))
